# Random Forest Model

This notebook trains a Random Forest model for cancellation prediction.

## Metrics Targets
- **F2-Score**: ≥ 0.68
- **Recall**: ≥ 70%
- **Precision**: ≥ 60%

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_recall_curve, average_precision_score,
    f1_score, fbeta_score, precision_score, recall_score,
    roc_auc_score, roc_curve
)
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# Load processed data
data_dir = "../data/silver"

X_train = pd.read_parquet(os.path.join(data_dir, "X_train.parquet"))
X_val = pd.read_parquet(os.path.join(data_dir, "X_val.parquet"))
X_test = pd.read_parquet(os.path.join(data_dir, "X_test.parquet"))

y_train = pd.read_parquet(os.path.join(data_dir, "y_train.parquet"))['is_cancelled']
y_val = pd.read_parquet(os.path.join(data_dir, "y_val.parquet"))['is_cancelled']
y_test = pd.read_parquet(os.path.join(data_dir, "y_test.parquet"))['is_cancelled']

print(f"X_train shape: {X_train.shape}")
print(f"Class distribution: {y_train.value_counts(normalize=True).to_dict()}")

In [ ]:
# Helper functions
def evaluate_model(y_true, y_pred, y_prob, dataset_name=""):
    f2 = fbeta_score(y_true, y_pred, beta=2)
    f1 = f1_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    pr_auc = average_precision_score(y_true, y_prob)
    roc_auc = roc_auc_score(y_true, y_prob)
    
    print(f"\n{'='*60}")
    print(f"EVALUATION - {dataset_name}")
    print(f"{'='*60}")
    print(f"F2-Score:  {f2:.4f} {'✅' if f2 >= 0.68 else '❌'}")
    print(f"Recall:    {recall:.4f} {'✅' if recall >= 0.70 else '❌'}")
    print(f"Precision: {precision:.4f} {'✅' if precision >= 0.60 else '❌'}")
    print(f"PR-AUC:    {pr_auc:.4f}")
    print(f"ROC-AUC:   {roc_auc:.4f}")
    
    return {'f2': f2, 'f1': f1, 'precision': precision, 'recall': recall, 'pr_auc': pr_auc, 'roc_auc': roc_auc}

def find_optimal_threshold(y_true, y_prob, beta=2):
    thresholds = np.arange(0.1, 0.9, 0.01)
    best_threshold, best_f_beta = 0.5, 0
    for thresh in thresholds:
        y_pred = (y_prob >= thresh).astype(int)
        f_beta = fbeta_score(y_true, y_pred, beta=beta)
        if f_beta > best_f_beta:
            best_f_beta = f_beta
            best_threshold = thresh
    return best_threshold, best_f_beta

## Train Random Forest

In [ ]:
# Train model with class weights
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)
print("Model trained.")

In [ ]:
# Get predictions
y_val_prob = model.predict_proba(X_val)[:, 1]
y_test_prob = model.predict_proba(X_test)[:, 1]

In [ ]:
# Find optimal threshold
optimal_threshold, best_f2 = find_optimal_threshold(y_val, y_val_prob)
print(f"Optimal threshold: {optimal_threshold:.2f}")
print(f"Best F2-Score: {best_f2:.4f}")

In [ ]:
# Evaluate with optimal threshold
y_val_pred = (y_val_prob >= optimal_threshold).astype(int)
y_test_pred = (y_test_prob >= optimal_threshold).astype(int)

val_metrics = evaluate_model(y_val, y_val_pred, y_val_prob, "Validation")
test_metrics = evaluate_model(y_test, y_test_pred, y_test_prob, "Test")

In [ ]:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nFeature Importance:")
print(feature_importance)

# Plot
fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(feature_importance['feature'], feature_importance['importance'], color='forestgreen')
ax.set_xlabel('Importance')
ax.set_title('Random Forest Feature Importance')
plt.tight_layout()
plt.show()

In [ ]:
# Save model
models_dir = "../models"
os.makedirs(models_dir, exist_ok=True)

model_artifacts = {
    'model': model,
    'optimal_threshold': optimal_threshold,
    'feature_names': list(X_train.columns),
    'metrics': {'validation': val_metrics, 'test': test_metrics}
}

with open(os.path.join(models_dir, "random_forest.pkl"), 'wb') as f:
    pickle.dump(model_artifacts, f)

print(f"Model saved to {models_dir}/random_forest.pkl")

In [ ]:
# Summary
print("\n" + "="*60)
print("RANDOM FOREST - FINAL SUMMARY")
print("="*60)
print(f"Optimal Threshold: {optimal_threshold:.2f}")
print(f"\nTest Set Performance:")
print(f"  F2-Score:  {test_metrics['f2']:.4f}")
print(f"  Recall:    {test_metrics['recall']:.4f}")
print(f"  Precision: {test_metrics['precision']:.4f}")